In [5]:
from datasets import load_from_disk

tokenized_dataset = load_from_disk("../data/tokenized_ncbi_disease")
print(tokenized_dataset)

label_names = ["O", "B-Disease", "I-Disease"]
num_labels = len(label_names)
print("Number of labels:", num_labels)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})
Number of labels: 3


In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=num_labels,
    id2label={0: "O", 1: "B-Disease", 2: "I-Disease"},
    label2id={"O": 0, "B-Disease": 1, "I-Disease": 2}
)

print("Model loaded!")
print("Model config:", model.config.num_labels, "labels")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!
Model config: 3 labels


In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters:     107,721,987
Trainable parameters: 107,721,987


In [8]:
import torch

# Take one sample from dataset
sample = tokenized_dataset["train"][0]

input_ids = torch.tensor([sample["input_ids"]])
attention_mask = torch.tensor([sample["attention_mask"]])
labels = torch.tensor([sample["labels"]])

# Forward pass
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)
print("Expected shape: [1, 512, 3] → [batch, tokens, num_labels]")

Loss: 1.0798280239105225
Logits shape: torch.Size([1, 512, 3])
Expected shape: [1, 512, 3] → [batch, tokens, num_labels]


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

# Get predictions from logits
predictions = torch.argmax(outputs.logits, dim=-1)
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])

print("Untrained model predictions on first sample:\n")
for token, pred, true_label in zip(tokens, predictions[0], labels[0]):
    if true_label != -100:  # only show real tokens
        pred_name = label_names[pred.item()]
        true_name = label_names[true_label.item()]
        match = "✅" if pred_name == true_name else "❌"
        print(f"{token:20} → Predicted: {pred_name:12} True: {true_name:12} {match}")

Untrained model predictions on first sample:

identification       → Predicted: B-Disease    True: O            ❌
of                   → Predicted: B-Disease    True: O            ❌
a                    → Predicted: B-Disease    True: O            ❌
,                    → Predicted: B-Disease    True: O            ❌
a                    → Predicted: B-Disease    True: O            ❌
ho                   → Predicted: O            True: O            ✅
of                   → Predicted: O            True: O            ✅
the                  → Predicted: B-Disease    True: O            ❌
ad                   → Predicted: B-Disease    True: B-Disease    ✅
p                    → Predicted: O            True: I-Disease    ❌
co                   → Predicted: O            True: I-Disease    ❌
t                    → Predicted: B-Disease    True: I-Disease    ❌
suppress             → Predicted: B-Disease    True: O            ❌
.                    → Predicted: B-Disease    True: O            ❌


The untrained model is just guessing I-Disease for almost everything.
This makes complete sense. The model has no idea what diseases look like yet — it just initialized its classification head with random weights. Since I-Disease is label index 2, and the random weights happen to favor that output, it's predicting it everywhere. This is called a random baseline

This is why training matters.
After fine-tuning on 5433 samples the model will learn:

Most tokens are O — the majority class
B-Disease only appears at the start of a disease entity
I-Disease only follows B-Disease, never appears alone

Right now it knows none of this.

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/biobert-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    logging_steps=50,
    fp16=False
)

print("Training arguments set!")

Training arguments set!


In [11]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_label_seq = []
        true_pred_seq = []

        for pred, label in zip(pred_seq, label_seq):
            if label != -100:  # ignore padded tokens
                true_label_seq.append(label_names[label])
                true_pred_seq.append(label_names[pred])

        true_labels.append(true_label_seq)
        true_predictions.append(true_pred_seq)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

print("Metric function ready!")

Metric function ready!


In [12]:
from transformers import DataCollatorForTokenClassification
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

print("Data collator ready!")

Data collator ready!


In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized!")
print("Training samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))

/var/folders/g_/t3vxvwv97ws7g69vh5cwr3fw0000gn/T/ipykernel_58609/1449757954.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer initialized!
Training samples: 5433
Validation samples: 924


In [14]:
steps_per_epoch = len(tokenized_dataset["train"]) // 16  # batch size 16
total_steps = steps_per_epoch * 3  # 3 epochs

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total training steps: {total_steps}")
print(f"Estimated time on CPU: ~{total_steps * 4 // 60} minutes (very slow)")
print(f"Estimated time on GPU: ~{total_steps * 2 // 60} minutes (fast)")
print("\nRecommendation: Run Cell 11 on Google Colab with GPU")

Steps per epoch: 339
Total training steps: 1017
Estimated time on CPU: ~67 minutes (very slow)
Estimated time on GPU: ~33 minutes (fast)

Recommendation: Run Cell 11 on Google Colab with GPU
